In [2]:
import os
import cv2
import time
import glob
import numpy as np
from pathlib import Path
from sklearn.cluster import KMeans
from skimage.segmentation import slic
from skimage.feature import local_binary_pattern
from sklearn.metrics import precision_score, recall_score, f1_score, jaccard_score

DATASET_ROOT = r"A:\wendang\comp9517\EWS-Dataset"
OUTPUT_ROOT = r"A:\wendang\comp9517\results\traditional_ablation"

N_SEGMENTS = 300
COMPACTNESS = 10
SIGMA = 1.0
LBP_RADIUS = 1
LBP_POINTS = 8 * LBP_RADIUS
LBP_METHOD = "uniform"
MORPH_KERNEL_SIZE = 5
MIN_COMPONENT_AREA = 80
SEEDS = [0, 1, 2, 3, 4]

FEATURE_MODES = ["F0", "F1", "F2", "F3"]
POST_MODES = ["P0", "P1", "P2", "P3"]
ROBUSTNESS_CONFIG = {
    "noise": [10, 20, 30],
    "blur": [3, 5, 7],
    "dark": [0.8, 0.6, 0.4]
}

def ensure_dir(p):
    os.makedirs(p, exist_ok=True)

def read_img(p):
    img = cv2.imread(p, cv2.IMREAD_COLOR)
    if img is None:
        raise FileNotFoundError(p)
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

def read_mask(p):
    mask = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
    if mask is None:
        raise FileNotFoundError(p)
    return (mask > 127).astype(np.uint8)

def pair(folder):
    files = glob.glob(os.path.join(folder, "*.png"))
    imgs = []
    masks = {}
    for f in files:
        name = Path(f).stem
        if name.endswith("_mask"):
            masks[name[:-5]] = f
        else:
            imgs.append((name, f))
    pairs = [(img, masks[name]) for name, img in imgs if name in masks]
    if len(pairs) == 0:
        raise RuntimeError(f"No pairs found in {folder}")
    return sorted(pairs, key=lambda x: x[0])

def get_data():
    train = pair(os.path.join(DATASET_ROOT, "train"))
    val = pair(os.path.join(DATASET_ROOT, "validation"))
    test = pair(os.path.join(DATASET_ROOT, "test"))
    return train, val, test

def exg(img):
    x = img.astype(np.float32) / 255.0
    return 2.0 * x[:, :, 1] - x[:, :, 0] - x[:, :, 2]

def features(img, seg, mode="F3"):
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    lbp = local_binary_pattern(gray, LBP_POINTS, LBP_RADIUS, method=LBP_METHOD)
    e = exg(img)
    n = int(seg.max()) + 1
    feats = []

    for i in range(n):
        m = (seg == i)
        if np.sum(m) == 0:
            if mode == "F0":
                feats.append([0.0, 0.0, 0.0])
            elif mode == "F1":
                feats.append([0.0, 0.0, 0.0, 0.0])
            elif mode == "F2":
                feats.append([0.0, 0.0, 0.0, 0.0])
            else:
                feats.append([0.0, 0.0, 0.0, 0.0, 0.0])
            continue

        f = [
            float(np.mean(img[:, :, 0][m])),
            float(np.mean(img[:, :, 1][m])),
            float(np.mean(img[:, :, 2][m]))
        ]

        if mode in ["F1", "F3"]:
            f.append(float(np.mean(e[m])))

        if mode in ["F2", "F3"]:
            f.append(float(np.mean(lbp[m])))

        feats.append(f)

    feats = np.array(feats, dtype=np.float32)
    feats = np.nan_to_num(feats, nan=0.0, posinf=0.0, neginf=0.0)

    mu = np.mean(feats, axis=0, keepdims=True)
    sd = np.std(feats, axis=0, keepdims=True)
    sd[sd < 1e-6] = 1.0
    feats = (feats - mu) / sd
    feats = np.nan_to_num(feats, nan=0.0, posinf=0.0, neginf=0.0)
    return feats

def post(mask, mode="P3", min_component_area=MIN_COMPONENT_AREA):
    m = (mask * 255).astype(np.uint8)

    if mode in ["P1", "P3"]:
        k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (MORPH_KERNEL_SIZE, MORPH_KERNEL_SIZE))
        m = cv2.morphologyEx(m, cv2.MORPH_OPEN, k)
        m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, k)

    if mode in ["P2", "P3"]:
        num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(m, connectivity=8)
        cleaned = np.zeros_like(m)
        for i in range(1, num_labels):
            if stats[i, cv2.CC_STAT_AREA] >= min_component_area:
                cleaned[labels == i] = 255
        m = cleaned

    return (m > 127).astype(np.uint8)

def fallback_segment(img, post_mode="P3", min_component_area=MIN_COMPONENT_AREA):
    e = exg(img)
    thr = float(np.mean(e))
    pred = (e > thr).astype(np.uint8)
    return post(pred, post_mode, min_component_area)

def segment(img, seed, feat_mode="F3", post_mode="P3", min_component_area=MIN_COMPONENT_AREA):
    seg = slic(
        img.astype(np.float32) / 255.0,
        n_segments=N_SEGMENTS,
        compactness=COMPACTNESS,
        sigma=SIGMA,
        start_label=0
    )

    n_regions = int(seg.max()) + 1
    if n_regions < 2:
        return fallback_segment(img, post_mode, min_component_area)

    f = features(img, seg, feat_mode)
    if f.shape[0] < 2:
        return fallback_segment(img, post_mode, min_component_area)

    km = KMeans(n_clusters=2, random_state=seed, n_init=10)
    lab = km.fit_predict(f)

    e = exg(img)
    scores = []
    for i in range(2):
        ids = np.where(lab == i)[0]
        if len(ids) == 0:
            scores.append(-1e9)
            continue
        m = np.isin(seg, ids)
        if np.sum(m) == 0:
            scores.append(-1e9)
            continue
        scores.append(float(np.mean(e[m])))

    crop = int(np.argmax(scores))
    pred = np.zeros(seg.shape, dtype=np.uint8)
    for i in np.where(lab == crop)[0]:
        pred[seg == i] = 1

    return post(pred, post_mode, min_component_area)

def evaluate(gt, pred):
    g = gt.flatten()
    p = pred.flatten()
    return [
        precision_score(g, p, zero_division=0),
        recall_score(g, p, zero_division=0),
        f1_score(g, p, zero_division=0),
        jaccard_score(g, p, zero_division=0)
    ]

def distort(img, kind=None, severity=None, seed=0):
    if kind is None or severity is None:
        return img.copy()

    rng = np.random.default_rng(seed)

    if kind == "noise":
        noise = rng.normal(0, severity, img.shape)
        out = np.clip(img.astype(np.float32) + noise, 0, 255).astype(np.uint8)
        return out

    if kind == "blur":
        k = int(severity)
        if k % 2 == 0:
            k += 1
        return cv2.GaussianBlur(img, (k, k), 0)

    if kind == "dark":
        factor = float(severity)
        out = np.clip(img.astype(np.float32) * factor, 0, 255).astype(np.uint8)
        return out

    return img.copy()

def overlay(img, mask):
    out = img.copy()
    green = np.zeros_like(out)
    green[:, :, 1] = 255
    alpha = 0.4
    out[mask == 1] = (alpha * green[mask == 1] + (1 - alpha) * out[mask == 1]).astype(np.uint8)
    return out

def run(
    pairs,
    feat_mode="F3",
    post_mode="P3",
    min_component_area=MIN_COMPONENT_AREA,
    distort_kind=None,
    severity=None,
    save_dir=None,
    save_images=False
):
    if save_dir is not None:
        ensure_dir(save_dir)

    all_metrics = []

    for seed in SEEDS:
        seed_metrics = []

        for img_p, mask_p in pairs:
            img = read_img(img_p)
            img = distort(img, distort_kind, severity, seed)
            gt = read_mask(mask_p)

            t = time.time()
            pred = segment(
                img,
                seed=seed,
                feat_mode=feat_mode,
                post_mode=post_mode,
                min_component_area=min_component_area
            )
            dt = time.time() - t

            p, r, f, i = evaluate(gt, pred)
            seed_metrics.append([p, r, f, i, dt])

            if save_images and save_dir is not None and seed == SEEDS[0]:
                base = Path(img_p).stem
                cv2.imwrite(os.path.join(save_dir, base + "_pred.png"), (pred * 255).astype(np.uint8))
                ov = overlay(img, pred)
                cv2.imwrite(os.path.join(save_dir, base + "_overlay.png"), cv2.cvtColor(ov, cv2.COLOR_RGB2BGR))

        all_metrics.append(np.mean(seed_metrics, axis=0))

    all_metrics = np.array(all_metrics, dtype=np.float32)
    return np.mean(all_metrics, axis=0), np.std(all_metrics, axis=0)

def format_metrics(mean, std):
    return {
        "Precision": f"{mean[0]:.4f}±{std[0]:.4f}",
        "Recall": f"{mean[1]:.4f}±{std[1]:.4f}",
        "F1": f"{mean[2]:.4f}±{std[2]:.4f}",
        "IoU": f"{mean[3]:.4f}±{std[3]:.4f}",
        "Time": f"{mean[4]:.4f}±{std[4]:.4f}"
    }

def print_metrics(title, mean, std):
    print(title)
    print(f"Precision mean={mean[0]:.4f} std={std[0]:.4f}")
    print(f"Recall    mean={mean[1]:.4f} std={std[1]:.4f}")
    print(f"F1        mean={mean[2]:.4f} std={std[2]:.4f}")
    print(f"IoU       mean={mean[3]:.4f} std={std[3]:.4f}")
    print(f"Time      mean={mean[4]:.4f} std={std[4]:.4f}")
    print()

def save_table_txt(path, rows, headers):
    with open(path, "w", encoding="utf-8") as f:
        f.write("\t".join(headers) + "\n")
        for row in rows:
            f.write("\t".join(str(x) for x in row) + "\n")

def feature_ablation(test_pairs):
    rows = []
    for feat_mode in FEATURE_MODES:
        mean, std = run(
            test_pairs,
            feat_mode=feat_mode,
            post_mode="P3",
            save_dir=os.path.join(OUTPUT_ROOT, "feature_ablation", feat_mode),
            save_images=(feat_mode == "F3")
        )
        print_metrics(f"Feature Ablation - {feat_mode}", mean, std)
        fm = format_metrics(mean, std)
        rows.append([feat_mode, fm["Precision"], fm["Recall"], fm["F1"], fm["IoU"], fm["Time"]])
    save_table_txt(
        os.path.join(OUTPUT_ROOT, "feature_ablation_table.txt"),
        rows,
        ["Mode", "Precision", "Recall", "F1", "IoU", "Time"]
    )

def post_ablation(test_pairs):
    rows = []
    configs = [
        ("P0", "P0", MIN_COMPONENT_AREA),
        ("P1", "P1", MIN_COMPONENT_AREA),
        ("P2", "P2", MIN_COMPONENT_AREA),
        ("P3", "P3", MIN_COMPONENT_AREA),
        ("P3_area30", "P3", 30),
        ("P3_area80", "P3", 80),
    ]
    for name, post_mode, area in configs:
        mean, std = run(
            test_pairs,
            feat_mode="F3",
            post_mode=post_mode,
            min_component_area=area,
            save_dir=os.path.join(OUTPUT_ROOT, "post_ablation", name),
            save_images=(name == "P3")
        )
        print_metrics(f"Post Ablation - {name}", mean, std)
        fm = format_metrics(mean, std)
        rows.append([name, fm["Precision"], fm["Recall"], fm["F1"], fm["IoU"], fm["Time"]])
    save_table_txt(
        os.path.join(OUTPUT_ROOT, "post_ablation_table.txt"),
        rows,
        ["Mode", "Precision", "Recall", "F1", "IoU", "Time"]
    )

def robustness_experiment(test_pairs):
    rows = []
    baseline_mean, baseline_std = run(
        test_pairs,
        feat_mode="F0",
        post_mode="P0",
        save_dir=os.path.join(OUTPUT_ROOT, "robustness", "baseline_clean"),
        save_images=False
    )
    improved_mean, improved_std = run(
        test_pairs,
        feat_mode="F3",
        post_mode="P3",
        save_dir=os.path.join(OUTPUT_ROOT, "robustness", "improved_clean"),
        save_images=True
    )
    print_metrics("Robustness - baseline clean", baseline_mean, baseline_std)
    print_metrics("Robustness - improved clean", improved_mean, improved_std)

    rows.append(["baseline", "clean", "-", *format_metrics(baseline_mean, baseline_std).values()])
    rows.append(["improved", "clean", "-", *format_metrics(improved_mean, improved_std).values()])

    for kind, severities in ROBUSTNESS_CONFIG.items():
        for severity in severities:
            baseline_mean, baseline_std = run(
                test_pairs,
                feat_mode="F0",
                post_mode="P0",
                distort_kind=kind,
                severity=severity,
                save_dir=os.path.join(OUTPUT_ROOT, "robustness", f"baseline_{kind}_{severity}"),
                save_images=False
            )
            improved_mean, improved_std = run(
                test_pairs,
                feat_mode="F3",
                post_mode="P3",
                distort_kind=kind,
                severity=severity,
                save_dir=os.path.join(OUTPUT_ROOT, "robustness", f"improved_{kind}_{severity}"),
                save_images=False
            )

            print_metrics(f"Robustness - baseline {kind} {severity}", baseline_mean, baseline_std)
            print_metrics(f"Robustness - improved {kind} {severity}", improved_mean, improved_std)

            rows.append(["baseline", kind, severity, *format_metrics(baseline_mean, baseline_std).values()])
            rows.append(["improved", kind, severity, *format_metrics(improved_mean, improved_std).values()])

    save_table_txt(
        os.path.join(OUTPUT_ROOT, "robustness_table.txt"),
        rows,
        ["Variant", "Distortion", "Severity", "Precision", "Recall", "F1", "IoU", "Time"]
    )

def normal_test(test_pairs):
    mean, std = run(
        test_pairs,
        feat_mode="F3",
        post_mode="P3",
        save_dir=os.path.join(OUTPUT_ROOT, "normal_test"),
        save_images=True
    )
    print_metrics("Normal Test - F3 + P3", mean, std)
    fm = format_metrics(mean, std)
    save_table_txt(
        os.path.join(OUTPUT_ROOT, "normal_test_table.txt"),
        [["F3_P3", fm["Precision"], fm["Recall"], fm["F1"], fm["IoU"], fm["Time"]]],
        ["Mode", "Precision", "Recall", "F1", "IoU", "Time"]
    )

def main():
    ensure_dir(OUTPUT_ROOT)
    train_pairs, val_pairs, test_pairs = get_data()

    print("Pairs:")
    print("Train:", len(train_pairs))
    print("Validation:", len(val_pairs))
    print("Test:", len(test_pairs))
    print()

    normal_test(test_pairs)
    feature_ablation(test_pairs)
    post_ablation(test_pairs)
    robustness_experiment(test_pairs)

if __name__ == "__main__":
    main()

Pairs:
Train: 142
Validation: 24
Test: 24

Normal Test - F3 + P3
Precision mean=0.5303 std=0.0005
Recall    mean=0.3601 std=0.0007
F1        mean=0.4202 std=0.0004
IoU       mean=0.2932 std=0.0004
Time      mean=0.1672 std=0.0020

Feature Ablation - F0
Precision mean=0.5440 std=0.0000
Recall    mean=0.3649 std=0.0001
F1        mean=0.4275 std=0.0001
IoU       mean=0.2983 std=0.0001
Time      mean=0.1581 std=0.0013

Feature Ablation - F1
Precision mean=0.5253 std=0.0002
Recall    mean=0.3479 std=0.0007
F1        mean=0.4098 std=0.0006
IoU       mean=0.2845 std=0.0006
Time      mean=0.1627 std=0.0013

Feature Ablation - F2
Precision mean=0.5474 std=0.0001
Recall    mean=0.3657 std=0.0005
F1        mean=0.4302 std=0.0003
IoU       mean=0.3001 std=0.0003
Time      mean=0.1630 std=0.0006

Feature Ablation - F3
Precision mean=0.5303 std=0.0005
Recall    mean=0.3601 std=0.0007
F1        mean=0.4202 std=0.0004
IoU       mean=0.2932 std=0.0004
Time      mean=0.1672 std=0.0014

Post Ablation - P